# Exercise — Adjudicate Master-Data Matches & Set Survivorship

**Trailhead Provisions** has duplicate customers because marketing keys on email while loyalty
keys on `customer_id`. Resolution auto-merges the obvious (`score >= 0.85`) and rejects the obvious
(`score < 0.66`); the **ambiguous** band (`0.66–0.84` — *pairs that share a name and an email that
differs only by mail provider or a trailing digit*) needs your judgment. Run the resolution workflow,
adjudicate all **8** pairs in the ambiguous band (`0..7`), pass your survivorship rule, and document
your stewardship log. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, apply_adjudications

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

m = gc.resolve_customers()
print("auto-merge:", int((m.pairs.band == "auto-merge").sum()),
      "| ambiguous:", len(m.ambiguous),
      "| rejected:", int((m.pairs.band == "reject").sum()))
# Display all signal columns (phone, country, created_ts) to separate true duplicates from homonyms.
# You can also inspect full customer rows with gc.query("SELECT * FROM customer WHERE customer_id IN (...)").
m.ambiguous[["id_a", "name_a", "email_a", "phone_a", "country_a", "created_a",
             "id_b", "name_b", "email_b", "phone_b", "country_b", "created_b", "score"]]

## 1. Adjudicate each ambiguous pair & set survivorship
Decide `"accept"` (same person) or `"reject"` (different people / homonym) for every row index `0..7` in `m.ambiguous`, and specify your `survivor` rule (`"lowest_id"` for the loyalty system of record, or `"earliest_created"`). Inspect the signal columns (`phone`, `country`, `created_a`/`created_b`) above before deciding.

In [ ]:
# TODO(you): set "accept" or "reject" per ambiguous row index (0..7) and choose your survivor rule.
decisions = {
    # 0: "accept",
}
survivor = None  # TODO(you): e.g. "lowest_id" (loyalty system of record) or "earliest_created"
outcome = apply_adjudications(m, decisions, survivor=survivor)
print({k: v for k, v in outcome.items() if k != "merges"})
outcome["merges"]

## 2. Stewardship decision log
Replace the cell below with your log.

> **Your stewardship log here.** Address:
>
> - A per-pair decision table covering all 8 ambiguous pairs (`0..7`) with the corroborating signals (`email`, `phone`, `country`, `created_ts`) and rationale for each `"accept"` or `"reject"`.
> - Your survivorship rule (`survivor`) and which domain is the system of record.
> - The false-positive vs false-negative tradeoff (why over-merging homonyms is worse under GDPR).
> - The structural cross-domain fix (shared key) that removes the ambiguity at source.